# Dolphin Pipeline

Thin Colab/local entrypoint for ByteDance Dolphin-v2 OCR execution.
Reusable logic stays in `src/`; this notebook only prepares runtime, uploads samples,
runs the two-stage Dolphin pipeline, and exports artifacts.

Colab note: run from the top after pulling updates.
The setup and extraction cells clear cached `src.*` modules so artifacts are generated from the latest code.

In [ ]:
from importlib import import_module
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

IS_COLAB = 'google.colab' in sys.modules
REPO_URL = os.environ.get('NFSE_REPO_URL', 'https://github.com/LuizCarlosGoulart/DolpII.git')


def default_repo_root():
    if IS_COLAB:
        return Path('/content/nfse-extractor')
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
        nested = candidate / 'nfse-extractor'
        if (nested / 'src').is_dir() and (nested / 'notebooks').is_dir():
            return nested
    return cwd


REPO_ROOT = Path(os.environ.get('NFSE_REPO_ROOT', str(default_repo_root())))
PROJECT_ROOT = Path(os.environ.get('NFSE_PROJECT_ROOT', str(REPO_ROOT / 'nfse-extractor' if IS_COLAB else REPO_ROOT)))

CONFIG = {
    'clone_or_pull': os.environ.get('NFSE_CLONE_OR_PULL', '1') == '1',
    'bootstrap_runtime': os.environ.get('NFSE_BOOTSTRAP_RUNTIME', '1') == '1',
    'mount_drive': os.environ.get('NFSE_MOUNT_DRIVE', '0') == '1',
    'sample_path': os.environ.get('NFSE_SAMPLE_PATH', ''),
    # Dolphin-specific settings
    'model_path': os.environ.get('NFSE_DOLPHIN_MODEL_PATH') or None,
    'device': os.environ.get('NFSE_DOLPHIN_DEVICE') or None,
    'runtime_factory_path': 'src.engines.dolphin_runtime.load_dolphin_runtime',
    'raw_output_path': os.environ.get(
        'NFSE_DOLPHIN_RAW_OUTPUT',
        '/content/dolphin_raw_artifacts.json' if IS_COLAB
        else str(PROJECT_ROOT / 'artifacts' / 'dolphin_raw_artifacts.json'),
    ),
    'candidate_output_path': os.environ.get(
        'NFSE_DOLPHIN_CANDIDATE_OUTPUT',
        '/content/dolphin_field_candidates.json' if IS_COLAB
        else str(PROJECT_ROOT / 'artifacts' / 'dolphin_field_candidates.json'),
    ),
    'preview_limit': int(os.environ.get('NFSE_PREVIEW_LIMIT', '80')),
    # Cache: skip GPU inference for documents whose raw artifact JSON already
    # exists on disk.  Set NFSE_OCR_CACHE=0 to force re-extraction.
    'use_ocr_cache': os.environ.get('NFSE_OCR_CACHE', '1') == '1',
}


def reset_project_imports():
    """Force Python to import project modules from disk after git updates."""
    importlib.invalidate_caches()
    removed = 0
    for module_name in list(sys.modules):
        if module_name == 'src' or module_name.startswith('src.'):
            del sys.modules[module_name]
            removed += 1
    return removed


def project_git_head():
    git_cwd = PROJECT_ROOT if PROJECT_ROOT.exists() else REPO_ROOT
    result = subprocess.run(
        ['git', '-C', str(git_cwd), 'log', '-1', '--oneline'],
        check=False, text=True, capture_output=True,
    )
    if result.returncode != 0:
        return 'not a git checkout yet'
    return result.stdout.strip() or 'unknown'


print(f'IS_COLAB: {IS_COLAB}')
print(f'REPO_ROOT: {REPO_ROOT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'use_ocr_cache: {CONFIG["use_ocr_cache"]}')

In [ ]:
if IS_COLAB and CONFIG['clone_or_pull']:
    if (REPO_ROOT / '.git').exists():
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull'], check=True)
    elif not REPO_ROOT.exists() or (REPO_ROOT.is_dir() and not any(REPO_ROOT.iterdir())):
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
    else:
        raise FileExistsError(
            f'{REPO_ROOT} exists but is not a git checkout. '
            'Remove it or set NFSE_REPO_ROOT to another path.'
        )

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {PROJECT_ROOT}. '
        'Check NFSE_PROJECT_ROOT or the cloned repository layout.'
    )

project_root_str = str(PROJECT_ROOT)
while project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)

if IS_COLAB and CONFIG['mount_drive']:
    from google.colab import drive
    drive.mount('/content/drive')

if IS_COLAB and CONFIG['bootstrap_runtime']:
    # --dolphin installs torch, transformers, accelerate, qwen_vl_utils, opencv-python
    subprocess.run(
        ['bash', str(PROJECT_ROOT / 'scripts' / 'colab_bootstrap.sh'), '--dolphin'],
        check=True,
    )

removed_modules = reset_project_imports()
print(f'Runtime ready: {project_git_head()}')
print(f'Project module cache cleared: {removed_modules} module(s)')

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# File upload / discovery  â”€  supports single OR multiple PDFs/images
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
if IS_COLAB and not CONFIG['sample_path']:
    from google.colab import files

    print('Upload one or more PDF/image files (Ctrl+click to select multiple):')
    uploaded = files.upload()

    sample_paths_raw: list[Path] = []
    for _fname in uploaded:
        _candidate = Path('/content') / _fname
        if not _candidate.exists():
            _matches = (
                sorted(Path.cwd().glob(_fname))
                + sorted(Path('/content').rglob(_fname))
            )
            _candidate = _matches[-1] if _matches else _candidate
        sample_paths_raw.append(_candidate)
else:
    # NFSE_SAMPLE_PATH may be a comma-separated list  (e.g. 'a.pdf,b.pdf,c.pdf')
    _raw_value = CONFIG['sample_path']
    if _raw_value:
        sample_paths_raw = [
            Path(p.strip()).expanduser()
            for p in _raw_value.split(',')
            if p.strip()
        ]
    else:
        sample_paths_raw = []

if not sample_paths_raw:
    raise ValueError(
        'No files to process. '
        'Upload files in Colab or set NFSE_SAMPLE_PATH before running.'
    )

# Validate existence; warn and skip any missing paths
sample_paths: list[Path] = []
for _p in sample_paths_raw:
    if _p.exists():
        sample_paths.append(_p)
    else:
        print(f'[WARN] File not found, skipping: {_p}')

if not sample_paths:
    raise FileNotFoundError('None of the provided files exist.')

# Keep CONFIG['sample_path'] pointing to the first file (backward compat)
CONFIG['sample_path'] = str(sample_paths[0])
sample_path = sample_paths[0]   # single-path variable preserved

print(f'Files queued for processing: {len(sample_paths)}')
for _i, _p in enumerate(sample_paths, 1):
    print(f'  [{_i}] {_p.name}  ({_p.stat().st_size // 1024} KB)')

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Batch Dolphin extraction  â”€  processes every file in sample_paths
# Cache: if NFSE_OCR_CACHE=1 (default) and raw artifact JSON already exists
#        for a document, GPU inference is skipped entirely.
# Note: Dolphin runs on a single GPU â€” documents are processed sequentially.
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import gc
import traceback
from time import perf_counter

removed_modules = reset_project_imports()
print(f'Using project commit: {project_git_head()}')
print(f'Project module cache cleared before extraction: {removed_modules} module(s)')
print(f'OCR cache : {"on" if CONFIG["use_ocr_cache"] else "off"}')
print()

from src.engines import DolphinExtractionAdapter
from src.export import load_extracted_elements_json, write_extracted_elements_json
from src.ingestion import load_document
from src.preprocessing import preprocess_document

normalization_module = importlib.import_module('src.normalization')
ConfigDrivenOutputNormalizer = getattr(normalization_module, 'ConfigDrivenOutputNormalizer', None)

# â”€â”€ Load runtime factory (resolves dotted path from CONFIG) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_factory_path = CONFIG['runtime_factory_path']
_mod_name, _fn_name = _factory_path.rsplit('.', 1)
_runtime_factory = getattr(import_module(_mod_name), _fn_name)

# â”€â”€ Shared resources  â”€  instantiated ONCE, model loaded here â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
adapter = DolphinExtractionAdapter(
    runtime_factory=_runtime_factory,
    model_path=CONFIG['model_path'],
    device=CONFIG['device'],
)
normalizer = ConfigDrivenOutputNormalizer() if ConfigDrivenOutputNormalizer is not None else None

# â”€â”€ Output path helpers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_raw_base_dir = Path(CONFIG['raw_output_path']).parent
_batch_root   = _raw_base_dir / 'dolphin_batch'


def _output_paths(stem, single_file):
    """Return (raw_path, candidate_path) for one document."""
    if single_file:
        return Path(CONFIG['raw_output_path']), Path(CONFIG['candidate_output_path'])
    doc_dir = _batch_root / stem
    doc_dir.mkdir(parents=True, exist_ok=True)
    return (
        doc_dir / (stem + '_raw_artifacts.json'),
        doc_dir / (stem + '_field_candidates.json'),
    )


# â”€â”€ Batch loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
batch_results: list[dict] = []
_is_single = len(sample_paths) == 1

# Updated on each success so preview cells below always show the last doc.
artifacts = []
candidates = []
document = None
raw_output_path = None
candidate_output_path = None

for _file_idx, _cur_path in enumerate(sample_paths, 1):
    _stem = _cur_path.stem
    print('\n' + chr(9472) * 60)
    print(f'[{_file_idx}/{len(sample_paths)}] {_cur_path.name}')
    _t0 = perf_counter()

    try:
        # â”€â”€ Load â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        _doc = load_document(_cur_path)
        print(f'  document_id : {_doc.document_id}')

        _raw_path, _cand_path = _output_paths(_stem, _is_single)

        # â”€â”€ Dolphin inference (or cache hit) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        _cached = False
        if CONFIG['use_ocr_cache'] and _raw_path.exists():
            try:
                _doc_artifacts = load_extracted_elements_json(_raw_path)
                # Validate cache belongs to current document (element_id = docid:dolphin:idx)
                if _doc_artifacts:
                    _cached_doc_id = _doc_artifacts[0].element_id.split(':')[0]
                    if _cached_doc_id != str(_doc.document_id):
                        print('  [CACHE MISS] document_id mismatch - re-extracting')
                    else:
                        _cached = True
                        _page_count = max((e.page_number or 1) for e in _doc_artifacts) if _doc_artifacts else 0
                        print(f'  [CACHE] {len(_doc_artifacts)} elements  (GPU inference skipped)')
            except Exception:
                # Corrupted cache or doc mismatch - re-extract
                _cached = False

        if not _cached:
            _preprocessed = preprocess_document(_doc)
            _page_count   = _preprocessed.metadata['page_count']
            print(f'  pages       : {_page_count}')

            _doc_artifacts = adapter.extract_preprocessed(_preprocessed)
            print(f'  raw elements: {len(_doc_artifacts)}')

            # Free page images after extraction to keep VRAM/RAM lean
            del _preprocessed
            gc.collect()

            _raw_path = write_extracted_elements_json(_doc_artifacts, _raw_path)

        # â”€â”€ Field normalisation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        _doc_candidates = []
        if normalizer is not None:
            _doc_candidates = normalizer.normalize(_doc, _doc_artifacts)
            _cand_path.parent.mkdir(parents=True, exist_ok=True)
            _cand_path.write_text(
                json.dumps(
                    [c.model_dump(mode='json') for c in _doc_candidates],
                    indent=2,
                    ensure_ascii=False,
                ),
                encoding='utf-8',
            )
        else:
            _cand_path = None

        _elapsed_ms = round((perf_counter() - _t0) * 1000, 1)
        print(f'  candidates  : {len(_doc_candidates)}')
        print(f'  raw output  : {_raw_path}')
        print(f'  cand output : {_cand_path}')
        print(f'  elapsed     : {_elapsed_ms} ms  OK{"  [cache]" if _cached else ""}')

        batch_results.append({
            'index':          _file_idx,
            'file':           _cur_path.name,
            'status':         'ok',
            'cached':         _cached,
            'document_id':    _doc.document_id,
            'page_count':     _page_count,
            'raw_elements':   len(_doc_artifacts),
            'candidates':     len(_doc_candidates),
            'raw_path':       str(_raw_path),
            'candidate_path': str(_cand_path) if _cand_path else None,
            'elapsed_ms':     _elapsed_ms,
        })

        # Update shared preview variables (always = last successful document)
        document              = _doc
        artifacts             = _doc_artifacts
        candidates            = _doc_candidates
        raw_output_path       = _raw_path
        candidate_output_path = _cand_path

    except Exception as _exc:
        _elapsed_ms = round((perf_counter() - _t0) * 1000, 1)
        print(f'  [ERROR] {_exc}')
        traceback.print_exc()
        batch_results.append({
            'index':      _file_idx,
            'file':       _cur_path.name,
            'status':     'error',
            'error':      str(_exc),
            'elapsed_ms': _elapsed_ms,
        })
        gc.collect()
        continue   # one failure does not abort the batch

print('\n' + '=' * 60)
_n_ok    = sum(1 for r in batch_results if r['status'] == 'ok')
_n_cache = sum(1 for r in batch_results if r.get('cached'))
_n_err   = sum(1 for r in batch_results if r['status'] == 'error')
print(
    f'Batch complete  {_n_ok} ok ({_n_cache} from cache) / '
    f'{_n_err} failed / {len(sample_paths)} total'
)

if document is None:
    raise RuntimeError('Every document failed to process. See errors above.')

print()
print('Active document for preview cells:')
print(f'  document_id : {document.document_id}')
print(f'  media_type  : {document.media_type}')
print(f'  raw_output  : {raw_output_path}')
print(f'  candidates  : {candidate_output_path}')

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Batch results table  â”€  one row per processed document
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_sep = '-' * 96
print(f'  {"#":<4} {"File":<35} {"Status":<8} {"C?":<4} {"Pages":<6} {"Elements":<10} {"Cands":<8} {"ms":>7}')
print(_sep)
for _r in batch_results:
    _fname = _r['file'][:34]
    if _r['status'] == 'ok':
        _c = 'âœ“' if _r.get('cached') else ' '
        print(
            f'  {_r["index"]:<4} {_fname:<35} {"ok":<8} {_c:<4}'
            f'{_r["page_count"]:<6} {_r["raw_elements"]:<10} '
            f'{_r["candidates"]:<8} {_r["elapsed_ms"]:>7}'
        )
    else:
        _short_err = str(_r.get('error', '?'))[:40]
        print(
            f'  {_r["index"]:<4} {_fname:<35} {"FAILED":<8} {"":4}'
            f'{"--":<6} {"--":<10} {"--":<8} {_r["elapsed_ms"]:>7}   <- {_short_err}'
        )
print(_sep)
print('C? = loaded from cache (GPU inference skipped)')
if len(sample_paths) > 1:
    print(f'Output root: {_batch_root}')
else:
    print(f'Output: {raw_output_path}')

In [ ]:
print('Raw element preview')
for item in artifacts[: CONFIG['preview_limit']]:
    print(
        f'{item.text!r} | conf={item.confidence} | page={item.page_number} | bbox={item.bounding_box}'
    )

if candidates:
    print('\nField candidate preview')
    for candidate in candidates[: CONFIG['preview_limit']]:
        print(
            f'{candidate.field_name} = {candidate.value!r} | '
            f'conf={candidate.confidence} | '
            f'section={candidate.metadata.get("section_name")} | '
            f'label={candidate.metadata.get("label_text")}'
        )

In [ ]:
important_fields = {
    'nfse_number',
    'issue_date',
    'verification_code',
    'provider_name',
    'provider_document',
    'provider_municipal_registration',
    'provider_address',
    'provider_email',
    'provider_uf',
    'provider_phone',
    'recipient_name',
    'recipient_document',
    'recipient_municipal_registration',
    'recipient_address',
    'recipient_email',
    'recipient_uf',
    'recipient_phone',
    'service_description',
    'service_code',
    'operation_nature',
    'service_city',
    'gross_amount',
    'deductions_amount',
    'taxable_amount',
    'iss_rate',
    'iss_amount',
    'iss_withheld_amount',
    'net_amount',
    'unconditional_discount',
    'conditional_discount',
    'pis_withheld_amount',
    'cofins_withheld_amount',
    'csll_withheld_amount',
    'inss_withheld_amount',
    'ir_withheld_amount',
    'other_retentions_amount',
}

for candidate in candidates:
    if candidate.field_name in important_fields:
        print(
            f'{candidate.field_name} = {candidate.value!r} | '
            f'conf={candidate.confidence} | '
            f'section={candidate.metadata.get("section_name")} | '
            f'label={candidate.metadata.get("label_text")} | '
            f'review={candidate.metadata.get("review_status")} | '
            f'reasons={candidate.metadata.get("review_reasons")} | '
            f'line={candidate.metadata.get("line_text")}'
        )

In [ ]:
from collections import Counter

review_counts = Counter(candidate.metadata.get('review_status', 'unknown') for candidate in candidates)
print('Review status summary')
for status, count in sorted(review_counts.items()):
    print(status, count)

needs_review = [c for c in candidates if c.metadata.get('manual_review_required')]
if needs_review:
    print()
    print('Needs review')
    for candidate in needs_review:
        print(
            f'{candidate.field_name} = {candidate.value!r} | '
            f'conf={candidate.confidence} | '
            f'reasons={candidate.metadata.get("review_reasons")} | '
            f'line={candidate.metadata.get("line_text")}'
        )
else:
    print()
    print('No candidate-level review flags.')

In [ ]:
from collections import Counter

counts = Counter(candidate.field_name for candidate in candidates)
for field_name, count in sorted(counts.items()):
    print(field_name, count)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Bloco A: Resumo de cobertura  ─  uma linha por nota, campos críticos em colunas
# ─────────────────────────────────────────────────────────────────────────────
_CRITICAL_FIELDS = [
    'nfse_number',
    'verification_code',
    'issue_date',
    'provider_name',
    'provider_document',
    'recipient_name',
    'service_description',
    'gross_amount',
    'net_amount',
]

_ALL_IMPORTANT_FIELDS = {
    'nfse_number', 'issue_date', 'verification_code',
    'provider_name', 'provider_document', 'provider_municipal_registration',
    'provider_address', 'provider_email', 'provider_uf', 'provider_phone',
    'recipient_name', 'recipient_document', 'recipient_municipal_registration',
    'recipient_address', 'recipient_email', 'recipient_uf', 'recipient_phone',
    'service_description', 'service_code', 'operation_nature', 'service_city',
    'gross_amount', 'deductions_amount', 'taxable_amount',
    'iss_rate', 'iss_amount', 'iss_withheld_amount', 'net_amount',
    'unconditional_discount', 'conditional_discount',
    'pis_withheld_amount', 'cofins_withheld_amount', 'csll_withheld_amount',
    'inss_withheld_amount', 'ir_withheld_amount', 'other_retentions_amount',
}

_CRIT_ABBREV = ['num', 'cod', 'dat', 'pre', 'pDoc', 'tom', 'svc', 'brut', 'liq']

# ── Carrega todos os JSONs de candidatos do batch ────────────────────────
_doc_field_map: dict[int, dict[str, str]] = {}
for _r in batch_results:
    _idx = _r['index']
    if _r['status'] != 'ok' or not _r.get('candidate_path'):
        _doc_field_map[_idx] = {}
        continue
    _cpath = Path(_r['candidate_path'])
    if not _cpath.exists():
        _doc_field_map[_idx] = {}
        continue
    try:
        _doc_field_map[_idx] = {
            c['field_name']: c['value']
            for c in json.loads(_cpath.read_text(encoding='utf-8'))
        }
    except Exception:
        _doc_field_map[_idx] = {}

# ── Tabela ────────────────────────────────────────────────────────────────
_hdr = '  '.join(f'{a:>5}' for a in _CRIT_ABBREV)
_sep = '-' * 100
print(f'  {"#":<4} {"Arquivo":<35}  {_hdr}  {"crit":>4}  {"total":>5}')
print(_sep)

_total_crit_hits = 0
_n_docs = 0

for _r in batch_results:
    _idx = _r['index']
    _fname = _r['file'][:34]
    _fields = _doc_field_map.get(_idx, {})

    if _r['status'] != 'ok':
        print(f'  {_idx:<4} {_fname:<35}  FAILED')
        continue

    _hits = [_cf in _fields for _cf in _CRITICAL_FIELDS]
    _crit_hit = sum(_hits)
    _marks = '  '.join('  ✓' if h else '  ✗' for h in _hits)
    _total_hit = sum(1 for f in _ALL_IMPORTANT_FIELDS if f in _fields)
    _flag = ' ◄' if _crit_hit < len(_CRITICAL_FIELDS) else ''

    print(
        f'  {_idx:<4} {_fname:<35}  {_marks}'
        f'  {_crit_hit:>2}/{len(_CRITICAL_FIELDS)}'
        f'  {_total_hit:>3}/{len(_ALL_IMPORTANT_FIELDS)}{_flag}'
    )
    _total_crit_hits += _crit_hit
    _n_docs += 1

print(_sep)
_avg = _total_crit_hits / _n_docs if _n_docs else 0
print(f'  Cobertura crítica média: {_avg:.1f}/{len(_CRITICAL_FIELDS)}  ({_avg / len(_CRITICAL_FIELDS) * 100:.0f}%)')
print()
print('  Colunas: ' + ' | '.join(f'{a}={cf}' for a, cf in zip(_CRIT_ABBREV, _CRITICAL_FIELDS)))
print('  ◄ = um ou mais campos críticos ausentes')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Bloco B: Detalhe por nota  ─  altere DETAIL_INDEX para inspecionar qualquer doc
# ─────────────────────────────────────────────────────────────────────────────
DETAIL_INDEX = len(batch_results)  # ← troque pelo índice desejado (coluna # acima)

_ORDERED_FIELDS = [
    # Identificação
    ('nfse_number',                     'critical'),
    ('verification_code',               'critical'),
    ('issue_date',                      'critical'),
    # Prestador
    ('provider_name',                   'critical'),
    ('provider_document',               'critical'),
    ('provider_municipal_registration', 'optional'),
    ('provider_address',                'optional'),
    ('provider_phone',                  'optional'),
    ('provider_email',                  'optional'),
    ('provider_uf',                     'optional'),
    # Tomador
    ('recipient_name',                  'critical'),
    ('recipient_document',              'optional'),
    ('recipient_municipal_registration','optional'),
    ('recipient_address',               'optional'),
    ('recipient_phone',                 'optional'),
    ('recipient_email',                 'optional'),
    ('recipient_uf',                    'optional'),
    # Serviço
    ('service_description',             'critical'),
    ('service_code',                    'optional'),
    ('operation_nature',                'optional'),
    ('service_city',                    'optional'),
    # Valores
    ('gross_amount',                    'critical'),
    ('net_amount',                      'critical'),
    ('deductions_amount',               'optional'),
    ('taxable_amount',                  'optional'),
    ('iss_rate',                        'optional'),
    ('iss_amount',                      'optional'),
    ('iss_withheld_amount',             'optional'),
    ('unconditional_discount',          'optional'),
    ('conditional_discount',            'optional'),
    ('pis_withheld_amount',             'optional'),
    ('cofins_withheld_amount',          'optional'),
    ('csll_withheld_amount',            'optional'),
    ('inss_withheld_amount',            'optional'),
    ('ir_withheld_amount',              'optional'),
    ('other_retentions_amount',         'optional'),
]

_GROUPS = {
    'nfse_number': 'IDENTIFICAÇÃO', 'verification_code': 'IDENTIFICAÇÃO', 'issue_date': 'IDENTIFICAÇÃO',
    'provider_name': 'PRESTADOR', 'provider_document': 'PRESTADOR',
    'provider_municipal_registration': 'PRESTADOR', 'provider_address': 'PRESTADOR',
    'provider_phone': 'PRESTADOR', 'provider_email': 'PRESTADOR', 'provider_uf': 'PRESTADOR',
    'recipient_name': 'TOMADOR', 'recipient_document': 'TOMADOR',
    'recipient_municipal_registration': 'TOMADOR', 'recipient_address': 'TOMADOR',
    'recipient_phone': 'TOMADOR', 'recipient_email': 'TOMADOR', 'recipient_uf': 'TOMADOR',
    'service_description': 'SERVIÇO', 'service_code': 'SERVIÇO',
    'operation_nature': 'SERVIÇO', 'service_city': 'SERVIÇO',
}

def _group_of(fname):
    return _GROUPS.get(fname, 'VALORES')

_detail_result = next((r for r in batch_results if r['index'] == DETAIL_INDEX), None)

if _detail_result is None:
    print(f'[ERRO] Nenhum documento com índice {DETAIL_INDEX}.')
elif _detail_result['status'] != 'ok':
    print(f'[ERRO] Documento {DETAIL_INDEX} falhou na extração.')
else:
    _dfields = _doc_field_map.get(DETAIL_INDEX, {})

    # Carrega metadata de review do JSON de candidatos
    _dreview: dict[str, dict] = {}
    _dpath = _detail_result.get('candidate_path')
    if _dpath and Path(_dpath).exists():
        for _c in json.loads(Path(_dpath).read_text(encoding='utf-8')):
            _dreview[_c['field_name']] = _c.get('metadata', {})

    _n_crit_total = sum(1 for _, t in _ORDERED_FIELDS if t == 'critical')
    _n_crit_found = sum(1 for f, t in _ORDERED_FIELDS if t == 'critical' and f in _dfields)
    _n_found      = sum(1 for f, _ in _ORDERED_FIELDS if f in _dfields)

    print(f'Nota #{DETAIL_INDEX}: {_detail_result["file"]}')
    print(f'  Elementos : {_detail_result["raw_elements"]}  |  Candidatos: {_detail_result["candidates"]}')
    print(f'  Críticos  : {_n_crit_found}/{_n_crit_total}  |  Total: {_n_found}/{len(_ORDERED_FIELDS)}')
    print()

    _last_group = None
    for _fname, _ftype in _ORDERED_FIELDS:
        _grp = _group_of(_fname)
        if _grp != _last_group:
            print(f'  ── {_grp} {"─" * max(1, 46 - len(_grp))}')
            _last_group = _grp

        _crit = _ftype == 'critical'
        _tag  = '★' if _crit else ' '

        if _fname in _dfields:
            _val     = _dfields[_fname]
            _meta    = _dreview.get(_fname, {})
            _status  = _meta.get('review_status', '')
            _reasons = _meta.get('review_reasons') or []
            _rtag    = f'  ⚠ {", ".join(_reasons)}' if _status == 'needs_review' else ''
            print(f'  {_tag} ✓  {_fname:<44} = {repr(_val)}{_rtag}')
        else:
            _miss = '  ← AUSENTE' if _crit else ''
            print(f'  {_tag} ✗  {_fname:<44}{_miss}')

    print()
    print('  ★ = crítico   ✓ = extraído   ✗ = não extraído   ⚠ = needs_review')